# ZINB Parameter Recovery Test

Generates from a known ZINB distribution and verifies that TensorZINB recovers the true parameters.

Also confirms that sparse and dense inputs produce equivalent log-likelihoods.

**Model structure:**
- NB mean: `log(mu) = beta[0] + beta[1]*x1 + beta[2]*x2`\n
- Zero-inflation: `logit(P(structural zero)) = gamma * z1`  (distinct predictor)
- Using a separate covariate for inflation avoids the known intercept/pi identifiability trade-off.

In [1]:
import numpy as np
import scipy.sparse
from scipy.stats import nbinom, bernoulli
import statsmodels.api as sm
from tensorzinb.tensorzinb import TensorZINB

np.random.seed(42)
nobs = 10000

# --- True parameters ---
true_beta  = np.array([1.0, 2.0, -1.5])  # NB log-mean: intercept, x1, x2
true_theta = 2.0                           # NB dispersion
true_gamma = 1.5                           # inflation: logit(P(zero)) = gamma * z1

# --- Covariates ---
x1 = np.random.binomial(1, 0.6, size=nobs).astype(float)
x2 = np.random.uniform(size=nobs)
z1 = np.random.normal(size=nobs)          # distinct predictor for zero-inflation
X      = sm.add_constant(np.column_stack([x1, x2]))
X_infl = z1.reshape(-1, 1)               # no intercept: identifiable because z1 ≠ const

# --- Generate ZINB data ---
mu  = np.exp(X @ true_beta)
nb_y = nbinom.rvs(true_theta, true_theta / (true_theta + mu), random_state=42)

pi_i = true_gamma * z1
zero_prob = 1 / (1 + np.exp(-pi_i))      # sample-wise P(structural zero)
structural_zero = bernoulli.rvs(zero_prob, random_state=42)
y = nb_y * (1 - structural_zero)

print(f"Mean structural zero rate: {zero_prob.mean():.3f}")
print(f"Observed zero rate:        {np.mean(y == 0):.3f}  (includes NB zeros)")
print(f"Mean count: {y.mean():.2f},  Var: {y.var():.2f}")

Mean structural zero rate: 0.500
Observed zero rate:        0.589  (includes NB zeros)
Mean count: 3.36,  Var: 49.02


## Fit: dense inputs

In [2]:
model_dense = TensorZINB(y.reshape(-1, 1), X, exog_infl=X_infl)
result_dense = model_dense.fit(init_method="nb", epochs=5000)

w         = result_dense['weights']
fit_beta  = w['x_mu'].flatten()
fit_theta = float(np.exp(w['theta']).flatten()[0])
fit_gamma = float(w['x_pi'].flatten()[0])

def sigmoid(x): return 1 / (1 + np.exp(-x))

# derived quantities in natural units
true_derived = {
    'mu (x1=0, x2=0.5)':      np.exp(true_beta[0] + 0*true_beta[1] + 0.5*true_beta[2]),
    'rate ratio x1':           np.exp(true_beta[1]),
    'rate ratio x2 (per 1)':   np.exp(true_beta[2]),
    'theta (dispersion)':      true_theta,
    'P(zero | z1=+1)':         sigmoid(true_gamma *  1),
    'P(zero | z1= 0)':         sigmoid(true_gamma *  0),
    'P(zero | z1=-1)':         sigmoid(true_gamma * -1),
}
fit_derived = {
    'mu (x1=0, x2=0.5)':      np.exp(fit_beta[0] + 0*fit_beta[1] + 0.5*fit_beta[2]),
    'rate ratio x1':           np.exp(fit_beta[1]),
    'rate ratio x2 (per 1)':   np.exp(fit_beta[2]),
    'theta (dispersion)':      fit_theta,
    'P(zero | z1=+1)':         sigmoid(fit_gamma *  1),
    'P(zero | z1= 0)':         sigmoid(fit_gamma *  0),
    'P(zero | z1=-1)':         sigmoid(fit_gamma * -1),
}

print(f"{'':26s} {'True':>8s} {'Fitted':>8s}")
print("-" * 44)
for k in true_derived:
    print(f"  {k:24s} {true_derived[k]:8.3f} {fit_derived[k]:8.3f}")
print(f"\n  Converged in {result_dense['epochs']} epochs")

I0000 00:00:1773772072.513136  147134 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled


                               True   Fitted
--------------------------------------------
  mu (x1=0, x2=0.5)           1.284    1.105
  rate ratio x1               7.389    8.657
  rate ratio x2 (per 1)       0.223    0.225
  theta (dispersion)          2.000    2.025
  P(zero | z1=+1)             0.818    0.931
  P(zero | z1= 0)             0.500    0.500
  P(zero | z1=-1)             0.182    0.069

  Converged in 316 epochs


## Fit: sparse inputs — should match dense numerically

In [3]:
X_sparse      = scipy.sparse.csr_matrix(X)
X_infl_sparse = scipy.sparse.csr_matrix(X_infl)

model_sparse = TensorZINB(y.reshape(-1, 1), X_sparse, exog_infl=X_infl_sparse)
result_sparse = model_sparse.fit(init_method="nb", epochs=5000)

print(f"{'':20s} {'Dense':>10s} {'Sparse':>10s}")
print("-" * 42)
for i, (d, s) in enumerate(zip(result_dense['weights']['x_mu'].flatten(),
                                result_sparse['weights']['x_mu'].flatten())):
    print(f"  beta[{i}]{'(intercept)' if i==0 else f'(x{i})':10s} {d:>10.3f} {s:>10.3f}")

print(f"\nllf  dense: {result_dense['llf_total']:.4f}")
print(f"llf sparse: {result_sparse['llf_total']:.4f}")
print(f"llf   diff: {abs(result_dense['llf_total'] - result_sparse['llf_total']):.4f}")

                          Dense     Sparse
------------------------------------------
  beta[0](intercept)      0.845      0.845
  beta[1](x1)            2.158      2.158
  beta[2](x2)           -1.491     -1.491

llf  dense: -15542.1004
llf sparse: -15542.1043
llf   diff: 0.0039
